# 04 — Analyze results

Compare experiment runs from `outputs/runs/` using a **single configuration cell**.

Supports MAE/RMSE comparison, error histograms, prediction vs ground truth, residuals, and per-image debug visualizations.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.colab_bootstrap import setup_notebook_environment
from src.config import get_config
from src.experiments.notebook_helpers import (
    AnalyzeResultsConfig,
    filter_summary,
    load_comparison,
    plot_error_histogram,
    plot_mae_rmse_panel,
    plot_pred_vs_gt,
    plot_residuals,
    resolve_figures_dir,
    resolve_runs_root,
    run_analysis_plots,
    top_k_errors,
)
from src.experiments.results import load_all_runs, load_registry
from src.utils import setup_logging
from src.visualization import (
    build_image_context,
    visualize_batch,
    visualize_comparison,
    visualize_image,
    visualize_measurement,
)

REPO, STORAGE = setup_notebook_environment(mount_drive=True)
setup_logging()
cfg = get_config()
runs_root = resolve_runs_root()
print(f"Storage: {STORAGE} | runs: {runs_root}")

## Configuration

Edit this cell to control which runs to compare, which plots to generate, and which images to visualize.

In [ ]:
ANALYSIS_CFG = AnalyzeResultsConfig(
    # --- runs to include in aggregate comparison ---
    run_names=[
        "baseline_bbox_v1",
        "baseline_pca_v1",
        "baseline_skeleton_v1",
        "advanced_full_v1",
    ],
    inspect_run="baseline_bbox_v1",   # per-image plots & error tables
    compare_run="advanced_full_v1",   # optional second run for visualize_comparison

    metrics=["mae_mm", "rmse_mm"],

    # --- plot toggles ---
    plot_mae_comparison=True,
    plot_rmse_comparison=True,
    plot_error_histogram=True,
    plot_pred_vs_gt=True,
    plot_residual=True,
    plot_pivot_table=True,
    plot_per_image_viz=True,

    # --- per-image visualization ---
    top_k_worst=5,
    top_k_best=5,
    viz_image_ids=None,                # explicit IDs override top-k selection
    save_viz=False,                    # save to figures_dir instead of inline display
    figures_dir=cfg.outputs_figures / "analysis",
    histogram_bins=20,
)

## Load experiment registry

In [ ]:
registry = load_registry(runs_root / "experiments.csv")
all_runs = load_all_runs(runs_root)
full_summary = registry if len(registry) else all_runs
summary = filter_summary(full_summary, ANALYSIS_CFG.run_names)
summary

## Metrics table

In [ ]:
if len(summary):
    cols = ["run_name", "pipeline", "method", "perspective", "mae_mm", "rmse_mm", "n_samples"]
    display_cols = [c for c in cols if c in summary.columns]
    display(summary[display_cols].sort_values("mae_mm"))
else:
    print("No runs matched — run 03_run_experiments.ipynb first or update ANALYSIS_CFG.run_names")

## Plots (config-driven)

In [ ]:
if len(summary):
    saved = run_analysis_plots(summary, ANALYSIS_CFG, runs_root, show=not ANALYSIS_CFG.save_viz)
    if ANALYSIS_CFG.save_viz:
        print("Saved:", {k: str(v) for k, v in saved.items() if v})
else:
    print("No runs to plot.")

## Per-run inspection: best / worst predictions

In [ ]:
inspect_dir = runs_root / ANALYSIS_CFG.inspect_run
comparison = load_comparison(inspect_dir)

if comparison is not None:
    worst = top_k_errors(comparison, k=ANALYSIS_CFG.top_k_worst, worst=True)
    best = top_k_errors(comparison, k=ANALYSIS_CFG.top_k_best, worst=False)
    print(f"=== Worst {ANALYSIS_CFG.top_k_worst} — {ANALYSIS_CFG.inspect_run} ===")
    display(worst[[c for c in ["image_id", "length_mm", "predicted_length_mm", "error_mm", "abs_error_mm"] if c in worst.columns]])
    print(f"\n=== Best {ANALYSIS_CFG.top_k_best} — {ANALYSIS_CFG.inspect_run} ===")
    display(best[[c for c in ["image_id", "length_mm", "predicted_length_mm", "error_mm", "abs_error_mm"] if c in best.columns]])
else:
    print(f"No comparison.csv at {inspect_dir}")

## Per-image debug visualizations

In [ ]:
if ANALYSIS_CFG.plot_per_image_viz and comparison is not None:
    if ANALYSIS_CFG.viz_image_ids:
        viz_ids = [str(i) for i in ANALYSIS_CFG.viz_image_ids]
    else:
        worst_ids = top_k_errors(comparison, k=ANALYSIS_CFG.top_k_worst, worst=True)["image_id"].astype(str).tolist()
        viz_ids = worst_ids[:3]

    for image_id in viz_ids:
        print(f"--- {image_id} ---")
        ctx = build_image_context(image_id, run_dir=inspect_dir)
        if ANALYSIS_CFG.save_viz:
            out_dir = resolve_figures_dir(ANALYSIS_CFG) / ANALYSIS_CFG.inspect_run
            visualize_image(ctx, debug=True, save=True, output_dir=out_dir, show=False)
            visualize_measurement(ctx, debug=True, save=True, output_dir=out_dir, show=False)
        else:
            visualize_image(ctx, debug=True)
            visualize_measurement(ctx, debug=True)

    if ANALYSIS_CFG.compare_run:
        compare_dir = runs_root / ANALYSIS_CFG.compare_run
        ctx0 = build_image_context(viz_ids[0], run_dir=inspect_dir)
        visualize_comparison(ctx0, compare_run_dir=compare_dir, debug=True, save=ANALYSIS_CFG.save_viz)

    if ANALYSIS_CFG.save_viz:
        visualize_batch(viz_ids, run_dir=inspect_dir, save=True, show=False,
                          output_dir=resolve_figures_dir(ANALYSIS_CFG) / ANALYSIS_CFG.inspect_run)
else:
    print("Per-image viz skipped (disabled or no comparison data).")

## Side-by-side metric table (baseline vs advanced)

In [ ]:
if ANALYSIS_CFG.plot_pivot_table and len(summary):
    pivot = summary.pivot_table(
        index="method",
        columns="pipeline",
        values="mae_mm",
        aggfunc="min",
    )
    display(pivot)
else:
    print("Pivot table skipped.")

---

## Legacy: manual inspection

Pre-refactor cells kept for reference.

In [ ]:
# RUN_TO_INSPECT = "baseline_bbox_v1"
# run_dir = runs_root / RUN_TO_INSPECT
# errors = pd.read_csv(run_dir / "comparison.csv")
# display(errors.head(10))
# plot_error_histogram(errors, run_name=RUN_TO_INSPECT, bins=20)